# Assignment 1 Question 2 MNIST MLflow

Prerequisite: a local MLflow Tracking Server must already be running:

mlflow server --backend-store-uri sqlite:///mlflow.db \
    --default-artifact-root ./mlruns --host 0.0.0.0 --port 5000 --allowed-hosts "*" --cors-allowed-origins "http://localhost:5000, http://127.0.0.1:5000"

## Step 0 — Setup
Install dependencies (skip if already installed) and import libraries.

In [1]:
# !pip install mlflow scikit-learn pandas --quiet

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp")
print("Tracking URI:", mlflow.get_tracking_uri())

2026/08/27 19:20:31 INFO mlflow.tracking.fluent: Experiment with name 'mnist-mlp' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000


## Step 1 — The starter script (un-instrumented)
This is the "before" version — plain scikit-learn, no tracking at all. Run it once just to confirm it works.

In [4]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = mnist.data.astype("float32")
y = mnist.target.astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

X_train = X_train / 255
X_val = X_val / 255
X_test = X_test/255

def train_and_evaluate(learning_rate=0.001, hidden_layer_sizes=(128,),batch_size=64):
    model = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, learning_rate_init=learning_rate, batch_size=batch_size, max_iter=20, random_state=42)
    model.fit(X_train, y_train)
    train_loss = model.loss_curve_[-1]
    val_preds = model.predict(X_val)
    val_accuracy = accuracy_score(y_val, val_preds)
    return model, train_loss, val_accuracy

# Sanity check — no MLflow involved yet
_, train_loss, val_accuracy = train_and_evaluate()
print(f"train_loss={train_loss:.4f}", f"val_accuracy={val_accuracy:.4f}")

/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


train_loss=0.0109 val_accuracy=0.9759


## Step 2 & 3 — Instrument it: manual logging
Wrap training in `with mlflow.start_run():` and log parameters, metrics, and a tag.

In [9]:
def train_and_log(learning_rate=0.001, hidden_layer_sizes=(128,), batch_size=64, run_name=None):
    with mlflow.start_run(run_name=run_name):
        # --- parameters (at least 3) ---
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("batch_size", batch_size)

        model, train_loss, val_accuracy = train_and_evaluate(learning_rate, hidden_layer_sizes, batch_size)

        # --- metrics (at least 2) ---
        mlflow.log_metric("train_loss", train_loss)
        mlflow.log_metric("val_accuracy", val_accuracy)

        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(model, name="model", skops_trusted_types=["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"])

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  train_loss={train_loss:.4f}  val_accuracy={val_accuracy:.4f}")
        return run_id

baseline_run_id = train_and_log(0.001, (128,), 64, run_name="mlp-baseline")

/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run f52ad65bc28a47038ed072de0b6ab7b5  |  train_loss=0.0109  val_accuracy=0.9759
🏃 View run mlp-baseline at: http://localhost:5000/#/experiments/1/runs/f52ad65bc28a47038ed072de0b6ab7b5
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Step 4 — Sweep: 6 runs varying `learning rate` and `batch size`
Re-run with different values and log each as its own MLflow run.

In [11]:
sweep_run_ids = []
for lr in [0.001, 0.01]:
    for batch in [32, 64, 128]:
        rid = train_and_log(learning_rate=lr, hidden_layer_sizes=(128,), batch_size=batch,
                         run_name=f"mlp-lr-{lr}-batch-{batch}")
        sweep_run_ids.append(rid)

print("Sweep run IDs:", sweep_run_ids)

/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 993c1ab684be4cfa8fed3505ab4846ff  |  train_loss=0.0110  val_accuracy=0.9775
🏃 View run mlp-lr-0.001-batch-32 at: http://localhost:5000/#/experiments/1/runs/993c1ab684be4cfa8fed3505ab4846ff
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 006695b058cd4ad599fdb08fdee8769b  |  train_loss=0.0109  val_accuracy=0.9759
🏃 View run mlp-lr-0.001-batch-64 at: http://localhost:5000/#/experiments/1/runs/006695b058cd4ad599fdb08fdee8769b
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


...

Logged run 16633eb5eeb94aa49ef5aa951378b06f  |  train_loss=0.1314  val_accuracy=0.9594
🏃 View run mlp-lr-0.01-batch-32 at: http://localhost:5000/#/experiments/1/runs/16633eb5eeb94aa49ef5aa951378b06f
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 590ead9114e842f8a682c31e2396f2e7  |  train_loss=0.0792  val_accuracy=0.9661
🏃 View run mlp-lr-0.01-batch-64 at: http://localhost:5000/#/experiments/1/runs/590ead9114e842f8a682c31e2396f2e7
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 316719835a9a440bbd9d4a4c8f6a6aa1  |  train_loss=0.0490  val_accuracy=0.9689
🏃 View run mlp-lr-0.01-batch-128 at: http://localhost:5000/#/experiments/1/runs/316719835a9a440bbd9d4a4c8f6a6aa1
🧪 View experiment at: http://localhost:5000/#/experiments/1
Sweep run IDs: ['993c1ab684be4cfa8fed3505ab4846ff', '006695b058cd4ad599fdb08fdee8769b', '59fd4e5c0d024310af5a5436de0918ee', '16633eb5eeb94aa49ef5aa951378b06f', '590ead9114e842f8a682c31e2396f2e7', '316719835a9a440bbd9d4a4c8f6a6aa1']


## Step 5 — A 5th run using `mlflow.autolog()`
Compare what gets captured automatically vs. what you logged by hand above.

In [12]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="mlp-autolog"):
    model = MLPClassifier(hidden_layer_sizes=(128,), learning_rate_init=0.001, batch_size=64, max_iter=20, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    # autolog captures params + many metrics automatically; we can still add a custom one
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, preds))
    autolog_run_id = mlflow.active_run().info.run_id

print("Autolog run:", autolog_run_id)
mlflow.sklearn.autolog(disable=True)  # turn autolog back off for the rest of the notebook

/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/08/27 20:58:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/home/pg/Assignment_1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


🏃 View run mlp-autolog at: http://localhost:5000/#/experiments/1/runs/f7220695ceb34470ada42ee82dbf75ed
🧪 View experiment at: http://localhost:5000/#/experiments/1
Autolog run: f7220695ceb34470ada42ee82dbf75ed


## Step 6 — Find the best run with `mlflow.search_runs()`
No need to open the UI to find the winner — query it directly.

In [13]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp"],
    order_by=["metrics.val_accuracy DESC"],
)

display_cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.learning_rate", "params.hidden_layer_sizes", "params.batch_size", "metrics.train_loss", "metrics.val_accuracy"
)]
print(runs_df[display_cols].head(10).to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nBest run: {best_run['run_id']}  (val_accuracy={best_run['metrics.val_accuracy']:.4f})")

                          run_id  metrics.val_accuracy  metrics.train_loss params.hidden_layer_sizes params.learning_rate params.batch_size    tags.mlflow.runName
993c1ab684be4cfa8fed3505ab4846ff              0.977524            0.010971                    (128,)                0.001                32  mlp-lr-0.001-batch-32
006695b058cd4ad599fdb08fdee8769b              0.975905            0.010872                    (128,)                0.001                64  mlp-lr-0.001-batch-64
f52ad65bc28a47038ed072de0b6ab7b5              0.975905            0.010872                    (128,)                0.001                64           mlp-baseline
62ffe31cd7b44ecc98755cbd3e02ffa8              0.975905            0.010872                    (128,)                0.001                64           mlp-baseline
59fd4e5c0d024310af5a5436de0918ee              0.973333            0.008139                    (128,)                0.001               128 mlp-lr-0.001-batch-128
316719835a9a440bbd9d4a